# OceanTrace AI — SegFormer-B2 Marine Oil Spill Segmentation
## Cloud Dataset Integration & GPU Training (Refined Deep-SAR Oil Spill / SOS)
### Smart India Hackathon 2026 | SIH26143

---

### Project Overview
This notebook executes the training pipeline for **OceanTrace AI's** oil spill semantic segmentation engine using the **SegFormer-B2** architecture (`nvidia/mit-b2`).
The primary dataset used is the **Refined Deep-SAR Oil Spill (SOS)** dataset (`bakhtiyar2222/deep-sar-oil-spill-segmentation-refined`), containing:
- **6,455 Training Image–Mask Pairs**
- **1,615 Validation Image–Mask Pairs**
- **Sentinel-1 SAR Satellite Imagery** (high-contrast radar backscatter)
- **Pixel-Level Ground Truth Masks** (Binary: `0: Background / Sea`, `1: Oil Spill`)

### Key Objectives
1. **Cloud Dataset Inspection & Validation**: Verify pair alignment, image resolutions, bit depths, label sanity, and compute exact class distributions.
2. **Binary Vision Transformer (SegFormer-B2)**: Fine-tune `nvidia/mit-b2` with an All-MLP decoder for binary segmentation (`num_labels=2`).
3. **Class-Imbalance Compensation**: Combined Weighted Cross-Entropy + Soft Dice Loss to accurately segment thin oil sheens and heavy emulsion patches.
4. **Full GPU Acceleration**: AMP (FP16 mixed precision), gradient clipping, cosine learning-rate decay, safe batch sizing, and early stopping.
5. **Research-Grade Evaluation & Export**: Precision, Recall, F1, IoU, Dice coefficient, Confusion Matrix, and prediction overlays saved to `/kaggle/working/oceantrace_artifacts/`.


In [ ]:
# Section 0: Dependency Installation & Environment Check
import sys
import subprocess

required_packages = [
    "transformers>=4.35.0",
    "scikit-learn>=1.3.0",
    "opencv-python-headless>=4.8.0",
    "albumentations>=1.3.1",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
    "tqdm>=4.66.0",
    "Pillow>=9.5.0",
]

print("Verifying and installing required packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + required_packages)
print("Dependencies installed successfully.")


In [ ]:
# Section 1: GPU Verification & Hardware Acceleration Report
import os
import torch
import transformers

print("=" * 65)
print("             OCEANTRACE AI — HARDWARE VERIFICATION")
print("=" * 65)
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL ERROR: CUDA GPU accelerator is NOT detected!
"
        "Full SegFormer training cannot run on CPU.

"
        "To enable GPU on Kaggle:
"
        "  1. Go to the right sidebar -> 'Session options'.
"
        "  2. Click 'Accelerator' and select 'GPU T4 x2' or 'GPU P100'.

"
        "To enable GPU on Google Colab:
"
        "  1. Go to Menu -> 'Runtime' -> 'Change runtime type'.
"
        "  2. Select 'T4 GPU' or higher under Hardware accelerator."
    )

device = torch.device("cuda:0")
gpu_name = torch.cuda.get_device_name(0)
gpu_count = torch.cuda.device_count()
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"Primary GPU Device   : {gpu_name}")
print(f"GPU Device Count     : {gpu_count}")
print(f"Total VRAM Available : {total_vram_gb:.2f} GB")
print(f"CUDA Capability      : {torch.cuda.get_device_capability(0)}")
print("=" * 65)
print("GPU acceleration confirmed. Ready for SegFormer-B2 training.")


In [ ]:
# Section 2: Automated Dataset Discovery & Directory Inspection
import glob
from pathlib import Path

# Base search paths for Kaggle and Colab environments
CANDIDATE_ROOTS = [
    Path("/kaggle/input/deep-sar-oil-spill-segmentation-refined"),
    Path("/kaggle/input/deep-sar-oil-spill-segmentation-refined/deep-sar-oil-spill-segmentation-refined"),
    Path("/kaggle/input/refined-deep-sar-oil-spill"),
    Path("/kaggle/input"),
    Path("./data"),
    Path("../data"),
    Path("/content"),
]

def discover_dataset():
    print("Searching for Refined Deep-SAR Oil Spill dataset...")
    
    # Check candidates
    for root in CANDIDATE_ROOTS:
        if not root.exists():
            continue
        
        # Look for train and val image directories
        for train_img_cand in root.glob("**/train/images"):
            parent = train_img_cand.parent.parent
            val_img_cand = parent / "val" / "images"
            if val_img_cand.exists() or (train_img_cand.parent / "images").exists():
                print(f"Found dataset at: {parent}")
                return parent
                
        # Alternative pattern: images/ and masks/
        for img_cand in root.glob("**/images"):
            parent = img_cand.parent
            if (parent / "masks").exists():
                print(f"Found dataset directory at: {parent}")
                return parent
                
    # Fallback to any directory containing PNGs
    print("Direct pattern search completed. Checking recursive image directories...")
    png_dirs = set()
    for root in CANDIDATE_ROOTS:
        if root.exists():
            for p in root.glob("**/*.png"):
                png_dirs.add(p.parent)
                if len(png_dirs) >= 10:
                    break
    if png_dirs:
        print(f"Discovered potential data folders: {list(png_dirs)[:3]}")
    
    return Path("/kaggle/input/deep-sar-oil-spill-segmentation-refined")

DATASET_ROOT = discover_dataset()
print(f"Active Dataset Root: {DATASET_ROOT}")

# Print directory structure
print("
Directory layout:")
for item in sorted(DATASET_ROOT.rglob("*")):
    if item.is_dir() and not item.name.startswith("."):
        rel_path = item.relative_to(DATASET_ROOT)
        depth = len(rel_path.parts)
        if depth <= 3:
            png_count = len(list(item.glob("*.png"))) + len(list(item.glob("*.jpg")))
            print(f"{'  ' * depth}📁 {item.name}/ ({png_count} images)")


if not DATASET_ROOT.exists() or len(list(DATASET_ROOT.glob("*"))) == 0:
    raise FileNotFoundError(
        "\n" + "="*70 + "\n"
        "ERROR: Refined Deep-SAR Oil Spill Dataset is NOT attached to /kaggle/input!\n"
        "To attach the dataset:\n"
        "  1. Look at the right sidebar in the Kaggle Editor.\n"
        "  2. Click '+ Add Data'.\n"
        "  3. Search for: bakhtiyar2222/deep-sar-oil-spill-segmentation-refined\n"
        "  4. Click the 'Add' (+) button next to it.\n"
        "  5. Rerun the notebook.\n" + "="*70
    )



In [ ]:
# Section 3: Comprehensive Dataset Validation & Quality Report
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

def locate_splits(root: Path):
    """Locate train, val, and test splits with images and masks."""
    splits = {}
    
    # Standard format: train/images, train/masks, val/images, val/masks
    for split_name in ["train", "val", "test", "validation"]:
        img_dir = root / split_name / "images"
        msk_dir = root / split_name / "masks"
        if not img_dir.exists():
            # Check recursive
            matches = list(root.glob(f"**/{split_name}/images"))
            if matches:
                img_dir = matches[0]
                msk_dir = img_dir.parent / "masks"
                
        if img_dir.exists() and msk_dir.exists():
            canonical = "val" if split_name == "validation" else split_name
            splits[canonical] = {"images": img_dir, "masks": msk_dir}
            
    # Single root fallback: images/ and masks/
    if not splits:
        img_dir = root / "images"
        msk_dir = root / "masks"
        if img_dir.exists() and msk_dir.exists():
            splits["all"] = {"images": img_dir, "masks": msk_dir}
            
    return splits

SPLITS = locate_splits(DATASET_ROOT)
print(f"Identified splits: {list(SPLITS.keys())}")

def validate_split(split_name: str, paths: dict):
    img_dir = paths["images"]
    msk_dir = paths["masks"]
    
    image_files = sorted(list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg")))
    mask_files = sorted(list(msk_dir.glob("*.png")) + list(msk_dir.glob("*.jpg")))
    
    img_map = {p.stem: p for p in image_files}
    msk_map = {p.stem: p for p in mask_files}
    
    common_stems = sorted(set(img_map.keys()) & set(msk_map.keys()))
    missing_masks = len(img_map) - len(common_stems)
    missing_images = len(msk_map) - len(common_stems)
    
    # Sample inspection (up to 200 items for speed)
    sample_size = min(len(common_stems), 200)
    sample_stems = common_stems[:sample_size]
    
    shapes = set()
    modes = set()
    mask_unique_values = set()
    oil_pixel_count = 0
    bg_pixel_count = 0
    empty_masks = 0
    corrupt_files = 0
    
    for stem in tqdm(sample_stems, desc=f"Validating {split_name}"):
        try:
            with Image.open(img_map[stem]) as im:
                shapes.add(im.size)
                modes.add(im.mode)
            with Image.open(msk_map[stem]) as mk:
                mk_arr = np.array(mk)
                u = np.unique(mk_arr)
                mask_unique_values.update(u.tolist())
                # Oil spill pixels are values > 0
                oil_px = np.count_nonzero(mk_arr > 0)
                bg_px = mk_arr.size - oil_px
                oil_pixel_count += oil_px
                bg_pixel_count += bg_px
                if oil_px == 0:
                    empty_masks += 1
        except Exception:
            corrupt_files += 1
            
    total_sampled_px = oil_pixel_count + bg_pixel_count
    oil_percentage = (oil_pixel_count / max(1, total_sampled_px)) * 100.0
    
    report = {
        "split": split_name,
        "total_images": len(image_files),
        "total_masks": len(mask_files),
        "matched_pairs": len(common_stems),
        "missing_masks": missing_masks,
        "missing_images": missing_images,
        "image_resolutions": [list(s) for s in shapes],
        "image_modes": list(modes),
        "mask_unique_values": sorted(list(mask_unique_values)),
        "empty_mask_samples": empty_masks,
        "corrupt_files": corrupt_files,
        "oil_pixel_percentage_sampled": round(oil_percentage, 3),
        "background_pixel_percentage_sampled": round(100.0 - oil_percentage, 3),
    }
    return report, [img_map[s] for s in common_stems], [msk_map[s] for s in common_stems]

VALIDATION_REPORTS = {}
SPLIT_DATA = {}

for split_name, paths in SPLITS.items():
    report, imgs, msks = validate_split(split_name, paths)
    VALIDATION_REPORTS[split_name] = report
    SPLIT_DATA[split_name] = {"images": imgs, "masks": msks}
    print(f"
--- {split_name.upper()} SPLIT SUMMARY ---")
    print(f"Matched Pairs       : {report['matched_pairs']}")
    print(f"Resolutions         : {report['image_resolutions']}")
    print(f"Mask Pixel Values   : {report['mask_unique_values']}")
    print(f"Oil Spill Pixels    : {report['oil_pixel_percentage_sampled']:.2f}% (Sampled)")

# Save validation report
ARTIFACTS_DIR = Path("/kaggle/working/oceantrace_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
validation_path = ARTIFACTS_DIR / "dataset_validation_report.json"
with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(VALIDATION_REPORTS, f, indent=2)
print(f"
Saved dataset validation report to: {validation_path}")


In [ ]:
# Section 4: Model Hyperparameter & Architecture Configuration
from dataclasses import dataclass, field
from typing import List

@dataclass
class SegFormerConfig:
    # Model Architecture
    pretrained_model_name: str = "nvidia/mit-b2"
    num_classes: int = 2
    image_size: int = 512
    
    # Binary Semantic Classes
    class_names: List[str] = field(default_factory=lambda: ["Background / Sea", "Oil Spill"])
    
    # Class weights for loss calculation
    # Oil spill pixels are typically ~2-5% of satellite ocean scenes
    class_weights: List[float] = field(default_factory=lambda: [1.0, 5.0])
    
    # Optimization & Training
    batch_size: int = 8          # Safe batch size for 16GB VRAM (T4 / P100)
    epochs: int = 25
    learning_rate: float = 6e-5
    min_learning_rate: float = 1e-6
    weight_decay: float = 0.01
    warmup_epochs: int = 2
    gradient_accumulation_steps: int = 2  # Effective batch size = 16
    max_grad_norm: float = 1.0
    use_amp: bool = True
    early_stopping_patience: int = 6
    random_seed: int = 42
    
    # Artifact Output Directory
    output_dir: str = "/kaggle/working/oceantrace_artifacts"

cfg = SegFormerConfig()
os.makedirs(cfg.output_dir, exist_ok=True)
print("SegFormer Training Configuration:")
for k, v in vars(cfg).items():
    print(f"  {k:30s}: {v}")


In [ ]:
# Section 5: Robust PyTorch Dataset with Binary Mask Thresholding & Augmentations
import random
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.random_seed)

class DeepSAROilSpillDataset(Dataset):
    """
    PyTorch Dataset for SAR satellite oil-spill binary segmentation.
    Loads SAR grayscale or RGB images, applies photometric and spatial augmentations,
    normalizes with ImageNet statistics, and thresholds binary masks (0: Background, 1: Oil Spill).
    """
    def __init__(self, image_paths, mask_paths, image_size=512, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size
        self.augment = augment
        
        # Albumentations pipeline
        if self.augment:
            self.transform = A.Compose([
                A.Resize(image_size, image_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.08, rotate_limit=30, p=0.4),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(image_size, image_size),
            ])
            
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load image
        img_path = str(self.image_paths[idx])
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img, dtype=np.uint8)
        
        # 2. Load mask
        msk_path = str(self.mask_paths[idx])
        msk = Image.open(msk_path).convert("L")
        msk_np = np.array(msk, dtype=np.uint8)
        
        # 3. Binary mask thresholding: any value > 0 becomes Class 1 (Oil Spill)
        msk_binary = (msk_np > 0).astype(np.uint8)
        
        # 4. Apply transforms
        augmented = self.transform(image=img_np, mask=msk_binary)
        aug_img = augmented["image"].astype(np.float32) / 255.0
        aug_msk = augmented["mask"].astype(np.int64)
        
        # 5. ImageNet normalization
        norm_img = (aug_img - self.mean) / self.std
        
        # 6. Convert to PyTorch tensors: (3, H, W) and (H, W)
        tensor_img = torch.from_numpy(norm_img.transpose(2, 0, 1)).float()
        tensor_msk = torch.from_numpy(aug_msk).long()
        
        return {
            "pixel_values": tensor_img,
            "labels": tensor_msk,
            "filename": Path(img_path).name
        }

# Prepare train and validation dataloaders
if "train" in SPLIT_DATA and "val" in SPLIT_DATA:
    train_imgs, train_msks = SPLIT_DATA["train"]["images"], SPLIT_DATA["train"]["masks"]
    val_imgs, val_msks = SPLIT_DATA["val"]["images"], SPLIT_DATA["val"]["masks"]
else:
    # Split from available data
    all_imgs, all_msks = list(SPLIT_DATA.values())[0]["images"], list(SPLIT_DATA.values())[0]["masks"]
    n_val = int(len(all_imgs) * 0.15)
    train_imgs, val_imgs = all_imgs[:-n_val], all_imgs[-n_val:]
    train_msks, val_msks = all_msks[:-n_val], all_msks[-n_val:]

train_dataset = DeepSAROilSpillDataset(train_imgs, train_msks, image_size=cfg.image_size, augment=True)
val_dataset = DeepSAROilSpillDataset(val_imgs, val_msks, image_size=cfg.image_size, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoaders successfully initialized:")
print(f"  Training batches   : {len(train_loader)} (total {len(train_dataset)} samples)")
print(f"  Validation batches : {len(val_loader)} (total {len(val_dataset)} samples)")


In [ ]:
# Section 6: Combined Weighted Cross-Entropy & Soft Dice Loss
import torch.nn as nn
import torch.nn.functional as F

class CombinedDiceCELoss(nn.Module):
    """
    Combined Weighted Cross-Entropy + Multi-Class Soft Dice Loss.
    Addresses severe class imbalance in satellite imagery where oil slicks occupy
    a minor fraction of the total pixel area.
    """
    def __init__(self, class_weights=None, ce_weight=0.5, dice_weight=0.5, eps=1e-6):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.eps = eps
        if class_weights is not None:
            weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
            self.ce = nn.CrossEntropyLoss(weight=weights_tensor)
        else:
            self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        # Move weights to logits device
        if self.ce.weight is not None and self.ce.weight.device != logits.device:
            self.ce.weight = self.ce.weight.to(logits.device)
            
        # Bilinear interpolation of SegFormer logits (H/4, W/4) to target size (H, W)
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(logits, size=targets.shape[-2:], mode="bilinear", align_corners=False)
            
        ce_loss = self.ce(logits, targets)
        
        # Soft Dice calculation
        probs = F.softmax(logits, dim=1)
        num_classes = logits.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)
        dice_score = (2.0 * intersection + self.eps) / (cardinality + self.eps)
        dice_loss = 1.0 - torch.mean(dice_score)
        
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss

criterion = CombinedDiceCELoss(class_weights=cfg.class_weights)
print("CombinedDiceCELoss initialized with weights:", cfg.class_weights)


In [ ]:
# Section 7: SegFormer-B2 (MiT-B2) Vision Transformer Model Initialization
from transformers import SegformerForSemanticSegmentation, SegformerConfig

id2label = {i: name for i, name in enumerate(cfg.class_names)}
label2id = {name: i for i, name in id2label.items()}

print(f"Loading SegFormer-B2 architecture from {cfg.pretrained_model_name}...")
model = SegformerForSemanticSegmentation.from_pretrained(
    cfg.pretrained_model_name,
    num_labels=cfg.num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print(f"Model Backbone       : {cfg.pretrained_model_name}")
print(f"Segmentation Head    : All-MLP Decoder")
print(f"Target Output Classes: {cfg.num_classes} ({cfg.class_names})")
print(f"Total Parameters     : {total_params / 1e6:.2f} Million")
print(f"Trainable Parameters : {trainable_params / 1e6:.2f} Million")
print("=" * 60)


In [ ]:
# Section 8: Comprehensive Evaluation Metrics Engine (IoU, Dice, F1, Precision, Recall)
from sklearn.metrics import confusion_matrix

def compute_metrics(pred_masks: np.ndarray, true_masks: np.ndarray, num_classes=2):
    """
    Calculates detailed segmentation metrics across binary classes:
      - Class 0: Background / Sea
      - Class 1: Oil Spill
    """
    preds_flat = pred_masks.flatten()
    trues_flat = true_masks.flatten()
    
    cm = confusion_matrix(trues_flat, preds_flat, labels=list(range(num_classes)))
    
    # Per-class IoU and Dice
    ious = []
    dices = []
    precisions = []
    recalls = []
    f1s = []
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        
        # IoU
        union = tp + fp + fn
        iou = tp / union if union > 0 else 0.0
        ious.append(float(iou))
        
        # Dice
        dice = (2.0 * tp) / (2.0 * tp + fp + fn) if (2.0 * tp + fp + fn) > 0 else 0.0
        dices.append(float(dice))
        
        # Precision & Recall
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        precisions.append(float(prec))
        recalls.append(float(rec))
        f1s.append(float(f1))
        
    pixel_acc = float(np.diag(cm).sum() / max(1, cm.sum()))
    mean_iou = float(np.mean(ious))
    mean_dice = float(np.mean(dices))
    
    return {
        "pixel_accuracy": round(pixel_acc, 4),
        "mean_iou": round(mean_iou, 4),
        "mean_dice": round(mean_dice, 4),
        "oil_spill_iou": round(ious[1], 4),
        "oil_spill_dice": round(dices[1], 4),
        "oil_spill_precision": round(precisions[1], 4),
        "oil_spill_recall": round(recalls[1], 4),
        "oil_spill_f1": round(f1s[1], 4),
        "background_iou": round(ious[0], 4),
        "per_class_iou": [round(x, 4) for x in ious],
        "confusion_matrix": cm.tolist()
    }

print("Metrics calculation engine compiled.")


In [ ]:
# Section 9: Complete GPU Training Loop with AMP, Cosine Scheduler, & Checkpointing
import time
import math
from torch.cuda.amp import autocast, GradScaler
import torch.optim as optim

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999)
)

# Cosine Annealing Learning Rate Scheduler with Warmup
total_steps = len(train_loader) * cfg.epochs // cfg.gradient_accumulation_steps
warmup_steps = len(train_loader) * cfg.warmup_epochs // cfg.gradient_accumulation_steps

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(cfg.min_learning_rate / cfg.learning_rate, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler(enabled=cfg.use_amp)

# Tracking history
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "oil_spill_iou": [],
    "mean_iou": [],
    "oil_spill_f1": [],
    "pixel_accuracy": [],
    "lr": []
}

best_oil_iou = 0.0
patience_counter = 0
best_checkpoint_path = os.path.join(cfg.output_dir, "best_oilspill_segformer.pth")
last_checkpoint_path = os.path.join(cfg.output_dir, "last_oilspill_segformer.pth")
final_checkpoint_path = os.path.join(cfg.output_dir, "final_oilspill_segformer.pth")

print("Beginning SegFormer-B2 Training...")
print("-" * 75)

for epoch in range(1, cfg.epochs + 1):
    epoch_start_time = time.time()
    
    # ------------------
    # Training Step
    # ------------------
    model.train()
    running_train_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_loader):
        images = batch["pixel_values"].to(device, non_blocking=True)
        masks = batch["labels"].to(device, non_blocking=True)
        
        with autocast(enabled=cfg.use_amp):
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, masks)
            loss = loss / cfg.gradient_accumulation_steps
            
        scaler.scale(loss).backward()
        
        if (step + 1) % cfg.gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
        running_train_loss += loss.item() * cfg.gradient_accumulation_steps * images.size(0)
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    current_lr = scheduler.get_last_lr()[0]
    
    # ------------------
    # Validation Step
    # ------------------
    model.eval()
    running_val_loss = 0.0
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for batch in val_loader:
            images = batch["pixel_values"].to(device, non_blocking=True)
            masks = batch["labels"].to(device, non_blocking=True)
            
            with autocast(enabled=cfg.use_amp):
                outputs = model(pixel_values=images)
                val_loss = criterion(outputs.logits, masks)
                
            running_val_loss += val_loss.item() * images.size(0)
            
            # Upsample logits to mask dimensions for evaluation
            logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.append(preds)
            all_trues.append(masks.cpu().numpy())
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    all_preds_np = np.concatenate(all_preds, axis=0)
    all_trues_np = np.concatenate(all_trues, axis=0)
    
    metrics = compute_metrics(all_preds_np, all_trues_np, num_classes=cfg.num_classes)
    epoch_duration = time.time() - epoch_start_time
    
    # Record history
    history["epoch"].append(epoch)
    history["train_loss"].append(round(epoch_train_loss, 4))
    history["val_loss"].append(round(epoch_val_loss, 4))
    history["oil_spill_iou"].append(metrics["oil_spill_iou"])
    history["mean_iou"].append(metrics["mean_iou"])
    history["oil_spill_f1"].append(metrics["oil_spill_f1"])
    history["pixel_accuracy"].append(metrics["pixel_accuracy"])
    history["lr"].append(current_lr)
    
    print(
        f"Epoch [{epoch:02d}/{cfg.epochs:02d}] ({epoch_duration:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Oil IoU: {metrics['oil_spill_iou']:.4f} | "
        f"Oil F1: {metrics['oil_spill_f1']:.4f} | "
        f"mIoU: {metrics['mean_iou']:.4f} | "
        f"LR: {current_lr:.2e}"
    )
    
    # Save periodic last checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "config": vars(cfg),
        "class_names": cfg.class_names,
    }, last_checkpoint_path)
    
    # Save best checkpoint
    if metrics["oil_spill_iou"] > best_oil_iou:
        best_oil_iou = metrics["oil_spill_iou"]
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_oil_spill_iou": best_oil_iou,
            "best_val_miou": metrics["mean_iou"],
            "metrics": metrics,
            "config": vars(cfg),
            "class_names": cfg.class_names,
            "preprocessing": {
                "image_size": cfg.image_size,
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            }
        }, best_checkpoint_path)
        print(f"  >>> ⭐ NEW BEST MODEL SAVED! Oil Spill IoU: {best_oil_iou:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= cfg.early_stopping_patience:
            print(f"
Early stopping triggered after {cfg.early_stopping_patience} epochs without improvement.")
            break

print("-" * 75)
print(f"Training completed. Best Oil Spill IoU achieved: {best_oil_iou:.4f}")

# Save final checkpoint
import shutil
shutil.copyfile(best_checkpoint_path if os.path.exists(best_checkpoint_path) else last_checkpoint_path, final_checkpoint_path)
print(f'Final model checkpoint saved to: {final_checkpoint_path}')



In [ ]:
# Section 10: Final Evaluation & Artifact Serialization
# Save training metrics and history
history_path = os.path.join(cfg.output_dir, "training_history.json")
metrics_path = os.path.join(cfg.output_dir, "training_metrics.json")
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

# Save model config and metadata
config_path = os.path.join(cfg.output_dir, "model_config.json")
metadata_path = os.path.join(cfg.output_dir, "model_metadata.json")
metadata_dict = {
    "model_name": "SegFormer-B2-Marine-OilSpill",
    "architecture": "Hierarchical Mix Transformer (MiT-B2) + All-MLP Lightweight Decoder",
    "pretrained_model_name": cfg.pretrained_model_name,
    "task": "Binary Semantic Segmentation",
    "target_classes": {
        "0": "Background / Sea",
        "1": "Oil Spill"
    },
    "num_classes": cfg.num_classes,
    "input_resolution": [cfg.image_size, cfg.image_size],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "mask_encoding": {
        "sea": 0,
        "oil_spill": 1,
        "thresholding_rule": "mask >= 127.5 for [0, 255] masks, mask >= 0.5 for [0, 1] masks"
    },
    "dataset": {
        "name": "Refined Deep-SAR Oil Spill (SOS)",
        "kaggle_source": "bakhtiyar2222/deep-sar-oil-spill-segmentation-refined",
        "zenodo_source": "https://zenodo.org/records/15298010",
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
    },
    "training_configuration": vars(cfg),
    "best_validation_metrics": {
        "best_oil_spill_iou": best_oil_iou,
        "final_metrics": metrics
    }
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(vars(cfg), f, indent=2)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_dict, f, indent=2)

# Save label mapping
labels_path = os.path.join(cfg.output_dir, "label_mapping.json")
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

# Save final evaluation report
eval_report_path = os.path.join(cfg.output_dir, "evaluation_report.json")
final_report = {
    "model_name": "SegFormer-B2-Marine-OilSpill",
    "backbone": cfg.pretrained_model_name,
    "num_classes": cfg.num_classes,
    "class_names": cfg.class_names,
    "best_oil_spill_iou": best_oil_iou,
    "final_epoch_metrics": metrics,
    "dataset": "Refined Deep-SAR Oil Spill (bakhtiyar2222/deep-sar-oil-spill-segmentation-refined)",
    "training_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
}
with open(eval_report_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)

print(f"Artifacts successfully serialized:")
print(f"  - {best_checkpoint_path}")
print(f"  - {last_checkpoint_path}")
print(f"  - {history_path}")
print(f"  - {metrics_path}")
print(f"  - {metadata_path}")
print(f"  - {config_path}")
print(f"  - {labels_path}")
print(f"  - {eval_report_path}")


# Save training history CSV
import pandas as pd
history_df = pd.DataFrame(history)
history_csv_path = os.path.join(cfg.output_dir, "training_history.csv")
history_df.to_csv(history_csv_path, index=False)

# Save class mapping
class_mapping_path = os.path.join(cfg.output_dir, "class_mapping.json")
with open(class_mapping_path, "w", encoding="utf-8") as f:
    json.dump({"0": "Background / Sea", "1": "Oil Spill"}, f, indent=2)

# Generate detailed training_report.md
report_md_path = os.path.join(cfg.output_dir, "training_report.md")
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write(f"# OceanTrace AI - SegFormer-B2 Training Report\n\n"
            f"## Model Configuration\n"
            f"- Backbone: {cfg.pretrained_model_name}\n"
            f"- Task: Binary Semantic Segmentation\n"
            f"- Input Resolution: {cfg.image_size}x{cfg.image_size}\n"
            f"- Epochs: {cfg.epochs}\n\n"
            f"## Validation Performance\n"
            f"- Best Oil Spill IoU: {best_oil_iou:.4f}\n"
            f"- Mean IoU: {metrics['mean_iou']:.4f}\n"
            f"- Oil Spill Dice: {metrics['class_dices'][1]:.4f}\n"
            f"- Oil Spill Precision: {metrics['class_precisions'][1]:.4f}\n"
            f"- Oil Spill Recall: {metrics['class_recalls'][1]:.4f}\n"
            f"- Pixel Accuracy: {metrics['pixel_accuracy']:.4f}\n")
print(f"  - {history_csv_path}")
print(f"  - {class_mapping_path}")
print(f"  - {report_md_path}")



In [ ]:
# Section 11: Visualization Plots & Sample Prediction Overlays
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Training Curves Plot
plt.figure(figsize=(15, 5))

# Loss subplot
plt.subplot(1, 3, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", color="#0284c7", lw=2)
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", color="#dc2626", lw=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Combined Loss Curves")
plt.legend()
plt.grid(True, alpha=0.3)

# IoU subplot
plt.subplot(1, 3, 2)
plt.plot(history["epoch"], history["oil_spill_iou"], label="Oil Spill IoU", color="#ea580c", lw=2)
plt.plot(history["epoch"], history["mean_iou"], label="Mean IoU", color="#16a34a", lw=2)
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("IoU Performance Curves")
plt.legend()
plt.grid(True, alpha=0.3)

# F1 / Precision / Accuracy subplot
plt.subplot(1, 3, 3)
plt.plot(history["epoch"], history["oil_spill_f1"], label="Oil Spill F1", color="#9333ea", lw=2)
plt.plot(history["epoch"], history["pixel_accuracy"], label="Pixel Accuracy", color="#64748b", lw=2)
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("F1 & Accuracy Curves")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(cfg.output_dir, "training_curves.png")
plt.savefig(curves_path, dpi=200)
plt.show()
print(f"Saved training curves to: {curves_path}")

# 2. Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm_data = np.array(metrics["confusion_matrix"])
# Normalize row-wise (recall)
cm_norm = cm_data.astype('float') / cm_data.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=cfg.class_names,
    yticklabels=cfg.class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("Ground Truth Label")
plt.title("Normalized Confusion Matrix")
plt.tight_layout()
cm_path = os.path.join(cfg.output_dir, "confusion_matrix.png")
plt.savefig(cm_path, dpi=200)
plt.show()
print(f"Saved confusion matrix to: {cm_path}")

# 3. Sample Prediction Overlays (10 Validation Images)
samples_dir = os.path.join(cfg.output_dir, "prediction_samples")
os.makedirs(samples_dir, exist_ok=True)

model.eval()
sample_indices = np.random.choice(len(val_dataset), size=min(10, len(val_dataset)), replace=False)

for s_idx in sample_indices:
    sample = val_dataset[s_idx]
    img_t = sample["pixel_values"].unsqueeze(0).to(device)
    gt_mask = sample["labels"].numpy()
    fn = sample["filename"]
    
    with torch.no_grad():
        out = model(pixel_values=img_t)
        logits = F.interpolate(out.logits, size=(cfg.image_size, cfg.image_size), mode="bilinear", align_corners=False)
        pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
    # Reconstruct display image from ImageNet norm
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    disp_img = sample["pixel_values"].permute(1, 2, 0).numpy() * std + mean
    disp_img = np.clip(disp_img, 0, 1)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(disp_img)
    axes[0].set_title(f"SAR Input ({fn})")
    axes[0].axis("off")
    
    axes[1].imshow(gt_mask, cmap="Reds", vmin=0, vmax=1)
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")
    
    axes[2].imshow(pred_mask, cmap="Reds", vmin=0, vmax=1)
    axes[2].set_title("SegFormer-B2 Prediction")
    axes[2].axis("off")
    
    # Blended Overlay
    overlay = disp_img.copy()
    overlay[pred_mask == 1] = [0.85, 0.15, 0.15]  # Red overlay
    axes[3].imshow(overlay)
    axes[3].set_title("Overlay Blend")
    axes[3].axis("off")
    
    plt.tight_layout()
    plt.savefig(os.path.join(samples_dir, f"prediction_{Path(fn).stem}.png"), dpi=150)
    plt.close()

print(f"Sample prediction overlays saved to: {samples_dir}")


# Mirror prediction samples to validation_visualizations for standard artifact specification
val_vis_dir = os.path.join(cfg.output_dir, "validation_visualizations")
shutil.copytree(samples_dir, val_vis_dir, dirs_exist_ok=True)
print(f"Mirrored sample predictions to: {val_vis_dir}")



In [ ]:
# Section 12: Automated Packaging & Zip Archive Generation
import shutil

zip_base = "/kaggle/working/oceantrace_artifacts"
print("Packaging all model checkpoints, reports, curves, and prediction samples into zip archive...")
zip_path = shutil.make_archive(zip_base, "zip", cfg.output_dir)
archive_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print("=" * 65)
print("OCEANTRACE AI — TRAINING & PACKAGING COMPLETE")
print("=" * 65)
print(f"Archive Created : {zip_path}")
print(f"Archive Size    : {archive_size_mb:.2f} MB")
print("Contents in archive:")
for root, dirs, files in os.walk(cfg.output_dir):
    for f in sorted(files):
        p = os.path.join(root, f)
        sz_kb = os.path.getsize(p) / 1024
        rel = os.path.relpath(p, cfg.output_dir)
        print(f"  - {rel:35s} ({sz_kb:8.1f} KB)")
print("=" * 65)
print("Ready for download and backend integration!")


## Section 13: Deployment Instructions

To integrate the trained model with OceanTrace AI:
1. Download `oceantrace_artifacts.zip` or `best_oilspill_segformer.pth` directly from Kaggle's `/kaggle/working/`.
2. Place `best_oilspill_segformer.pth` in the project root or in `oceantrace_artifacts/best_oilspill_segformer.pth`.
3. Start the FastAPI backend:
   ```bash
   uvicorn app.main:app --port 8000
   ```
4. Query `GET /api/model/status` to verify that `checkpoint_loaded: true` and `model_status: "operational"`.
5. Run full inference via `POST /api/spill/segformer-detect` or on the live dashboard.
